# C-C network

In [ ]:
rho_list = [1.0]
theta_list = ['-0.8']
rho_str = '1.0' 

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys
import os

figure_name = 'png_elu_ECC3'

os.makedirs(figure_name, exist_ok=True)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys

plot_gap = 0.5

for rho in rho_list:
    for theta in theta_list:

        b_h = 2.25*rho
        b_v = 1.5*rho

        file_name = f"RBMSteadyState_extendedCrissCross_none_100_100_100_D3_{theta}_0.0_0.01_200_0.1_64_6000_dim=3_TD1_elu_50_addcost=0.05_0.2_0.3_lr140000_eta={rho_str}_pricing=Y_delay=2.25_1.5"
        model_path = f"../../../1_NN_training/4_ECC_saved_NNs/{file_name}_grad"
        model = tf.saved_model.load(model_path)

        plt.rcParams.update({'font.size': 28,
                           'text.usetex': True})

        font1 = {'family': 'Arial',
                'weight': 'normal',
                'size': 28}


        # plot single boundary control


        np.set_printoptions(precision=6)
        np.set_printoptions(suppress=True)
        np.set_printoptions(threshold=sys.maxsize)

        suffix = ""
        char2num = {str(i): i for i in range(10)}.__getitem__

        activation = "elu"

        cur_model_grad = keras.models.load_model("../../../1_NN_training/4_ECC_saved_NNs/" + file_name+"_grad")

        for bb in [b_h + b_v/2]:
            S_shrink = (np.minimum(bb,3)/3+1)/2
            S_shrink2 = np.minimum(bb,3)/3
            steps = 200
            plt.rcParams["figure.figsize"] = (7.5,7.5)
            plt.rcParams.update({'font.size': 28})
            plt.rc('legend',fontsize=28)
            x1list = np.linspace(-plot_gap, b_v + b_h + plot_gap, steps)
            x2list = np.linspace(-plot_gap, b_v + b_h + plot_gap, steps)
            x3list = np.array([bb])

            X1, X2, X3 = np.meshgrid(x1list, x2list, x3list)

            X1_f = X1.flatten()
            X2_f = X2.flatten()
            X3_f = X3.flatten()

            points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

            cur_grad = cur_model_grad(points)

            Z0 = cur_grad[:,0].numpy()
            Z1 = cur_grad[:,1].numpy()
            Z2 = cur_grad[:,2].numpy()

            fig = plt.figure(figsize = [14, 7])

            X1_mat = X1_f.reshape([len(x2list), len(x1list)])
            X2_mat = X2_f.reshape([len(x2list), len(x1list)])
            Z0_mat = Z0.reshape([len(x2list), len(x1list)])
            Z1_mat = Z1.reshape([len(x2list), len(x1list)])

            plt.subplot(1,2,1)

            CS = plt.contourf(X1_mat, X2_mat, Z0_mat, np.arange(-0.1, 1.1, 0.1), cmap='jet')
            plt.colorbar()

            x = np.linspace(0, (b_v+b_h)*S_shrink, steps)
            # mask outside the state space (white); overlap the strips to avoid seams
            x_mask = np.linspace(-plot_gap, b_v+b_h+plot_gap, steps)
            plt.fill_between(x_mask, -plot_gap, np.maximum(0, x_mask - b_v), facecolor='white', zorder=1.5)
            plt.fill_between(x_mask, (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], -plot_gap, 0, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            # plt.plot(x, np.maximum(0,x-b_v),'--w', linewidth=2.5)
            # plt.plot(x, np.ones(steps)*(b_v+b_h)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*0*S_shrink, np.linspace(0, b_h + b_v, 50)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*(b_v+b_h)*S_shrink, b_h*S_shrink2 + np.linspace(0, b_v, 50), '--w', linewidth=2.5)
            plt.xlabel(r'$w_1$')
            plt.ylabel(r'$w_2$')
            plt.xlim([0, b_v + b_h])
            plt.ylim([0, b_v + b_h])
            plt.xticks([0, 1, 2, 3])
            plt.xticks([0, 1, 2, 3])
            plt.title(r'Gradient along $w_1$')
            plt.tight_layout()

            plt.subplot(1,2,2)

            CS = plt.contourf(X1_mat, X2_mat, Z1_mat, np.arange(-0.1, 1.1, 0.1), cmap='jet')
            plt.colorbar()

            x = np.linspace(0, (b_v+b_h)*S_shrink, steps)
            # mask outside the state space (white); overlap the strips to avoid seams
            x_mask = np.linspace(-plot_gap, b_v+b_h+plot_gap, steps)
            plt.fill_between(x_mask, -plot_gap, np.maximum(0, x_mask - b_v), facecolor='white', zorder=1.5)
            plt.fill_between(x_mask, (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], -plot_gap, 0, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            # plt.plot(x, np.maximum(0,x-b_v),'--w', linewidth=2.5)
            # plt.plot(x, np.ones(steps)*(b_v+b_h)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*0*S_shrink, np.linspace(0, b_h + b_v, 50)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*(b_v+b_h)*S_shrink, b_h*S_shrink2 + np.linspace(0, b_v, 50), '--w', linewidth=2.5)
            plt.xlabel(r'$w_1$')
            plt.ylabel(r'$w_2$')
            plt.xlim([0, b_v + b_h])
            plt.ylim([0, b_v + b_h])
            plt.xticks([0, 1, 2, 3])
            plt.xticks([0, 1, 2, 3])
            plt.title(r'Gradient along $w_2$')
            plt.tight_layout()

            fig.suptitle(r'Slice: $w_3 = b_h + b_v/2$', fontsize=36)
            fig.tight_layout(rect=[0, 0, 1, 1.05])

            plt.savefig(f"{figure_name}/ECC3_NN_gradient_upper.png")
            plt.close()

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys

plot_gap = 0.5

for rho in rho_list:
    for theta in theta_list:

        b_h = 2.25*rho
        b_v = 1.5*rho

        file_name = f"RBMSteadyState_extendedCrissCross_none_100_100_100_D3_{theta}_0.0_0.01_200_0.1_64_6000_dim=3_TD1_elu_50_addcost=0.05_0.2_0.3_lr140000_eta={rho_str}_pricing=Y_delay=2.25_1.5"
        model_path = f"../../../1_NN_training/4_ECC_saved_NNs/{file_name}_grad"
        model = tf.saved_model.load(model_path)

        plt.rcParams.update({'font.size': 28,
                           'text.usetex': True})

        font1 = {'family': 'Arial',
                'weight': 'normal',
                'size': 28}


        # plot single boundary control


        np.set_printoptions(precision=6)
        np.set_printoptions(suppress=True)
        np.set_printoptions(threshold=sys.maxsize)

        suffix = ""
        char2num = {str(i): i for i in range(10)}.__getitem__

        activation = "elu"

        cur_model_grad = keras.models.load_model("../../../1_NN_training/4_ECC_saved_NNs/" + file_name+"_grad")

        for bb in [b_v/2]:
            S_shrink = 1
            S_shrink2 = 1

            steps = 200
            plt.rcParams["figure.figsize"] = (7.5,7.5)
            plt.rcParams.update({'font.size': 28})
            plt.rc('legend',fontsize=28)
            x1list = np.array([bb])
            x2list = np.linspace(-plot_gap, b_v + b_h + plot_gap, steps)
            x3list = np.linspace(-plot_gap, b_v + b_h + plot_gap, steps)

            X1, X2, X3 = np.meshgrid(x1list, x2list, x3list)

            X1_f = X1.flatten()
            X2_f = X2.flatten()
            X3_f = X3.flatten()

            points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

            cur_grad = cur_model_grad(points)

            Z0 = cur_grad[:,0].numpy()
            Z1 = cur_grad[:,1].numpy()
            Z2 = cur_grad[:,2].numpy()

            fig = plt.figure(figsize = [14, 7])

            X2_mat = X2_f.reshape([len(x3list), len(x2list)])
            X3_mat = X3_f.reshape([len(x3list), len(x2list)])
            Z1_mat = Z1.reshape([len(x3list), len(x2list)])
            Z2_mat = Z2.reshape([len(x3list), len(x2list)])

            plt.subplot(1,2,1)

            CS = plt.contourf(X2_mat, X3_mat, Z1_mat, np.arange(-0.1, 1.1, 0.1), cmap='jet')
            plt.colorbar()

            x = np.linspace(0, (b_v+b_h)*S_shrink, steps)
            # mask outside the state space (white); overlap the strips to avoid seams
            x_mask = np.linspace(-plot_gap, b_v+b_h+plot_gap, steps)
            plt.fill_between(x_mask, -plot_gap, np.maximum(0, x_mask - b_v), facecolor='white', zorder=1.5)
            plt.fill_between(x_mask, (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], -plot_gap, 0, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            # plt.plot(x, np.maximum(0,x-b_v),'--w', linewidth=2.5)
            # plt.plot(x, np.ones(steps)*(b_v+b_h)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*0*S_shrink, np.linspace(0, b_h + b_v, 50)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*(b_v+b_h)*S_shrink, b_h*S_shrink2 + np.linspace(0, b_v, 50), '--w', linewidth=2.5)
            plt.xlabel(r'$w_2$')
            plt.ylabel(r'$w_3$')
            plt.xlim([0, b_v + b_h])
            plt.ylim([0, b_v + b_h])
            plt.xticks([0, 1, 2, 3])
            plt.xticks([0, 1, 2, 3])
            plt.title(r'Gradient along $w_2$')
            plt.tight_layout()

            plt.subplot(1,2,2)

            CS = plt.contourf(X2_mat, X3_mat, Z2_mat, np.arange(-0.1, 1.1, 0.1), cmap='jet')
            plt.colorbar()

            x = np.linspace(0, (b_v+b_h)*S_shrink, steps)
            # mask outside the state space (white); overlap the strips to avoid seams
            x_mask = np.linspace(-plot_gap, b_v+b_h+plot_gap, steps)
            plt.fill_between(x_mask, -plot_gap, np.maximum(0, x_mask - b_v), facecolor='white', zorder=1.5)
            plt.fill_between(x_mask, (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], -plot_gap, 0, facecolor='white', zorder=1.5)
            plt.fill_betweenx([-plot_gap, b_v+b_h+plot_gap], (b_v+b_h)*S_shrink, b_v+b_h+plot_gap, facecolor='white', zorder=1.5)
            # plt.plot(x, np.maximum(0,x-b_v),'--w', linewidth=2.5)
            # plt.plot(x, np.ones(steps)*(b_v+b_h)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*0*S_shrink, np.linspace(0, b_h + b_v, 50)*S_shrink, '--w', linewidth=2.5)
            # plt.plot(np.ones(50)*(b_v+b_h)*S_shrink, b_h*S_shrink2 + np.linspace(0, b_v, 50), '--w', linewidth=2.5)
            plt.xlabel(r'$w_2$')
            plt.ylabel(r'$w_3$')
            plt.xlim([0, b_v + b_h])
            plt.ylim([0, b_v + b_h])
            plt.xticks([0, 1, 2, 3])
            plt.xticks([0, 1, 2, 3])
            plt.title(r'Gradient along $w_3$')
            plt.tight_layout()

            fig.suptitle(r'Slice: $w_1 = b_v/2$', fontsize=36)
            fig.tight_layout(rect=[0, 0, 1, 1.05])

            plt.savefig(f"{figure_name}/ECC3_NN_gradient_lower.png")
            plt.close()